In [1]:
import sqlite3

conexao = sqlite3.connect('/content/vagas_tech.db')
cursor = conexao.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

conexao.close()

[]


In [2]:
!wget https://dot.net/v1/dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

--2026-09-14 00:27:31--  https://dot.net/v1/dotnet-install.sh
Resolving dot.net (dot.net)... 20.76.201.171, 20.112.250.133, 20.236.44.162, ...
Connecting to dot.net (dot.net)|20.76.201.171|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh [following]
--2026-09-14 00:27:32--  https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh
Resolving builds.dotnet.microsoft.com (builds.dotnet.microsoft.com)... 23.1.254.204, 23.1.254.208, 2600:1407:7400:82::173e:b64f, ...
Connecting to builds.dotnet.microsoft.com (builds.dotnet.microsoft.com)|23.1.254.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/octet-stream]
Saving to: ‘dotnet-install.sh’

dotnet-install.sh       [ <=>                ]  62.07K  --.-KB/s    in 0.01s   

2026-09-14 00:27:32 (4.35 MB/s) - ‘dotnet-install.sh’ saved [63563]

dotnet-install: Attempting

In [3]:
import os
os.environ["PATH"] += ":/root/.dotnet"

In [4]:
!dotnet new console -n VagasTechApp

=========
Welcome to .NET 8.0!
---------------------
SDK Version: 8.0.425

Telemetry
---------
The .NET tools collect usage data in order to help us improve your experience. It is collected by Microsoft and shared with the community. You can opt-out of telemetry by setting the DOTNET_CLI_TELEMETRY_OPTOUT environment variable to '1' or 'true' using your favorite shell.

Read more about .NET CLI Tools telemetry: https://aka.ms/dotnet-cli-telemetry

----------------
Installed an ASP.NET Core HTTPS development certificate.
To trust the certificate, view the instructions: https://aka.ms/dotnet-https-linux

----------------
Write your first app: https://aka.ms/dotnet-hello-world
Find out what's new: https://aka.ms/dotnet-whats-new
Explore documentation: https://aka.ms/dotnet-docs
Report issues and find source on GitHub: https://github.com/dotnet/core
Use 'dotnet --help' to see available commands or visit: https://aka.ms/dotnet-cli
----------------------------------------------------

In [5]:
%cd /content/VagasTechApp

/content/VagasTechApp


In [6]:
!dotnet add package Microsoft.Data.Sqlite --version 8.0.11

=  Determining projects to restore...
  Writing /tmp/tmpR7ml2u.tmp
=info : X.509 certificate chain validation will use the fallback certificate bundle at '/root/.dotnet/sdk/8.0.425/trustedroots/codesignctl.pem'.
info : X.509 certificate chain validation will use the fallback certificate bundle at '/root/.dotnet/sdk/8.0.425/trustedroots/timestampctl.pem'.
info : Adding PackageReference for package 'Microsoft.Data.Sqlite' into project '/content/VagasTechApp/VagasTechApp.csproj'.
info : Restoring packages for /content/VagasTechApp/VagasTechApp.csproj...
info :   GET https://api.nuget.org/v3-flatcontainer/microsoft.data.sqlite/index.json
info :   OK https://api.nuget.org/v3-flatcontainer/microsoft.data.sqlite/index.json 53ms
info :   GET https://api.nuget.org/v3-flatcontainer/microsoft.data.sqlite/8.0.11/microsoft.data.sqlite.8.0.11.nupkg
info :   OK https://api.nuget.org/v3-flatcontainer/microsoft.data.sqlite/8.0.11/microsoft.data.sqlite.8.0.11.nupkg 62ms
info :   GET https://api.nuget.

In [42]:
%%writefile MetodosCRUD.cs
using System;
using Microsoft.Data.Sqlite;

namespace VagasTechApp;

public static class MetodosCRUD {
    private const string ConnectionString = "Data Source=/content/vagas_tech.db";

    public static void CadastrarVaga(int idVaga, string titulo, string empresa, decimal salario) {

        using var conexao = new SqliteConnection(ConnectionString);
        conexao.Open();
        using var comando = conexao.CreateCommand();

        comando.CommandText = """
                INSERT INTO VAGAS (ID_VAGA, TITULO, EMPRESA, SALARIO)
                VALUES (@idVaga, @titulo, @empresa, @salario);
                """;

        comando.Parameters.AddWithValue("@idVaga", idVaga);
        comando.Parameters.AddWithValue("@titulo", titulo);
        comando.Parameters.AddWithValue("@empresa", empresa);
        comando.Parameters.AddWithValue("@salario", salario);

        comando.ExecuteNonQuery();

        Console.WriteLine($"✅ Vaga '{titulo}' cadastrada com sucesso!");
    }

    public static void CadastrarCandidata(int idCandidata, string nome, string email) {

        using var conexao = new SqliteConnection(ConnectionString);
        conexao.Open();
        using var comando = conexao.CreateCommand();

        comando.CommandText = """
                INSERT INTO CANDIDATAS (ID_CANDIDATA, NOME, EMAIL)
                VALUES (@idCandidata, @nome, @email);
                """;

        comando.Parameters.AddWithValue("@idCandidata", idCandidata);
        comando.Parameters.AddWithValue("@nome", nome);
        comando.Parameters.AddWithValue("@email", email);

        comando.ExecuteNonQuery();

        Console.WriteLine($"✅ Candidata '{nome}' cadastrada com sucesso!");
    }

    public static void EnviarCandidatura(int idCandidatura, DateTime dataEnvio, int idVaga, int idCandidata) {

        using var conexao = new SqliteConnection(ConnectionString);
        conexao.Open();
        using var comando = conexao.CreateCommand();

        comando.CommandText = """
                INSERT INTO CANDIDATURAS (ID_CANDIDATURA, DATA_ENVIO, ID_VAGA, ID_CANDIDATA)
                VALUES (@idCandidatura, @dataEnvio, @idVaga, @idCandidata);
                """;

        comando.Parameters.AddWithValue("@idCandidatura", idCandidatura);
        comando.Parameters.AddWithValue("@dataEnvio", dataEnvio);
        comando.Parameters.AddWithValue("@idVaga", idVaga);
        comando.Parameters.AddWithValue("@idCandidata", idCandidata);

        comando.ExecuteNonQuery();

        Console.WriteLine("✅ Candidatura enviada com sucesso!");
    }

    public static void ConsultarCandidaturas() {

        string query = """
                SELECT
                    c.NOME,
                    c.EMAIL,
                    v.TITULO,
                    v.EMPRESA
                FROM CANDIDATURAS cd
                INNER JOIN CANDIDATAS c ON cd.ID_CANDIDATA = c.ID_CANDIDATA
                INNER JOIN VAGAS v ON cd.ID_VAGA = v.ID_VAGA;
                """;

        using var connection = new SqliteConnection(ConnectionString);
        connection.Open();

        using var command = new SqliteCommand(query, connection);

        using var reader = command.ExecuteReader();

        Console.WriteLine("\n=== CONSULTA DE CANDIDATURAS (READ) ===");

        if (!reader.HasRows) {
            Console.WriteLine("🚨 Nenhuma candidatura encontrada.");
            return;
        }

        while (reader.Read()) {
            string nome = reader.GetString(0);
            string email = reader.GetString(1);
            string titulo = reader.GetString(2);
            string empresa = reader.GetString(3);

            Console.WriteLine($"Candidata: {nome} ({email}) | Vaga: {titulo} - Empresa: {empresa}");
        }
    }

    public static void AtualizarSalarioVaga(int idVaga, decimal novoSalario) {

        using var conexao = new SqliteConnection(ConnectionString);
        conexao.Open();
        using var comando = conexao.CreateCommand();

        comando.CommandText = """
                        UPDATE VAGAS
                        SET SALARIO = @novoSalario
                        WHERE ID_VAGA = @idVaga;
                        """;

        comando.Parameters.AddWithValue("@novoSalario", novoSalario);
        comando.Parameters.AddWithValue("@idVaga", idVaga);

        int linhasAfetadas = comando.ExecuteNonQuery();

        if (linhasAfetadas > 0) {
            Console.WriteLine("✅ Salário da vaga atualizado com sucesso!");
        }
        else {
            Console.WriteLine("Vaga não encontrada.");
        }
    }

    public static void CancelarCandidatura(int idCandidatura) {

        using var conexao = new SqliteConnection(ConnectionString);
        conexao.Open();

        using var comando = conexao.CreateCommand();

        comando.CommandText = """
                                DELETE FROM CANDIDATURAS
                                WHERE ID_CANDIDATURA = @idCandidatura;
                                """;

        comando.Parameters.AddWithValue("@idCandidatura", idCandidatura);

        int linhasAfetadas = comando.ExecuteNonQuery();

        if (linhasAfetadas > 0) {
            Console.WriteLine("✅ Candidatura cancelada com sucesso!");
        }
        else {
            Console.WriteLine("Candidatura não encontrada.");
        }
    }
}

Overwriting MetodosCRUD.cs


In [50]:
%%writefile Program.cs
using System;
using VagasTechApp;

Console.WriteLine("\n================================================");
Console.WriteLine(" INICIANDO INTEGRAÇÃO DA PLATAFORMA VAGASTECH...");
Console.WriteLine("================================================");

try
{
    // --- ETAPA 1: CREATE (Cadastrar VAGAS) ---

    Console.WriteLine("\n[Etapa 1] Cadastrando Vagas no sistema...\n");
    MetodosCRUD.CadastrarVaga(1, "Engenheira de Dados", "Alfa", 9000);
    MetodosCRUD.CadastrarVaga(2, "Analista de BI", "Ômega", 7500);

    // --- ETAPA 2: CREATE (Cadastrar CANDIDATA) ---

    Console.WriteLine("\n[Etapa 2] Cadastrando Candidata no sistema...\n");
    MetodosCRUD.CadastrarCandidata(1, "Mariana Souza", "mariana.souza@gmail.com");

    // --- ETAPA 3: CREATE (Enviar DUAS CANDIDATURAS) ---

    Console.WriteLine("\n[Etapa 3] Enviando duas candidaturas...\n");
    MetodosCRUD.EnviarCandidatura(901, DateTime.Now, 1, 1);
    MetodosCRUD.EnviarCandidatura(902, DateTime.Now, 2, 1);

    // --- ETAPA 4: READ (Consultar Candidaturas) ---

    Console.WriteLine("\n[Etapa 4] Consultando lista de candidaturas...");
    MetodosCRUD.ConsultarCandidaturas();

    // --- ETAPA 5: UPDATE (Atualizar salário da vaga 1) ---

    Console.WriteLine("\n[Etapa 5] Salário da vaga 'Engenheira de Dados' mudou. Atualizando...\n");
    MetodosCRUD.AtualizarSalarioVaga(1, 9500);

    // --- ETAPA 6: READ (Confirmar Atualização) ---

    Console.WriteLine("\n[Etapa 6] Consultando lista de candidaturas novamente para confirmar alteração...");
    MetodosCRUD.ConsultarCandidaturas();

    // --- ETAPA 7: DELETE (Cancelar apenas a inscrição 902) ---

    Console.WriteLine("\n[Etapa 7] Aluna cancelou a inscrição 902. Removendo...\n");
    MetodosCRUD.CancelarCandidatura(902);

    // --- ETAPA 8: READ (Confirmar Cancelamento e Ver o que Sobrou) ---

    Console.WriteLine("\n[Etapa 8] Consultando a lista final para validação...");
    MetodosCRUD.ConsultarCandidaturas();
}
catch (Exception ex)
{
    Console.WriteLine($"🚨 Ocorreu um erro no pipeline de integração: {ex.Message}");
}
finally
{
    Console.WriteLine("\n================================================");
    Console.WriteLine("        PROCESSO DE INTEGRAÇÃO FINALIZADO.      ");
    Console.WriteLine("================================================");
}

Overwriting Program.cs


In [43]:
!dotnet build

=  Determining projects to restore...
  All projects are up-to-date for restore.
  VagasTechApp -> /content/VagasTechApp/bin/Debug/net8.0/VagasTechApp.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:01.54


In [51]:
!dotnet run

==
 INICIANDO INTEGRAÇÃO DA PLATAFORMA VAGASTECH...

[Etapa 1] Cadastrando Vagas no sistema...

✅ Vaga 'Engenheira de Dados' cadastrada com sucesso!
✅ Vaga 'Analista de BI' cadastrada com sucesso!

[Etapa 2] Cadastrando Candidata no sistema...

✅ Candidata 'Mariana Souza' cadastrada com sucesso!

[Etapa 3] Enviando duas candidaturas...

✅ Candidatura enviada com sucesso!
✅ Candidatura enviada com sucesso!

[Etapa 4] Consultando lista de candidaturas...

=== CONSULTA DE CANDIDATURAS (READ) ===
Candidata: Mariana Souza (mariana.souza@gmail.com) | Vaga: Engenheira de Dados - Empresa: Alfa
Candidata: Mariana Souza (mariana.souza@gmail.com) | Vaga: Analista de BI - Empresa: Ômega

[Etapa 5] Salário da vaga 'Engenheira de Dados' mudou. Atualizando...

✅ Salário da vaga atualizado com sucesso!

[Etapa 6] Consultando lista de candidaturas novamente para confirmar alteração...

=== CONSULTA DE CANDIDATURAS (READ) ===
Candidata: Mariana Souza (mariana.souza@gmail.com) | Vaga: Engenheira de Dad